# Notebook 008. Fold and bootstrap-unit assignment
-------

Assign each labelled parcel a spatial cross-validation fold and a bootstrap unit, then report balance, cross-unit separation and fold buffer sensitivity. One labelled-parcel block bootstrap serves both performance and predicted-area uncertainty.

In [ ]:
# Setup. Load the reference labels, derive the class grouping and old-growth flag,
# isolate the labelled parcels, and report the connected-component structure that
# the fold partition groups on.
import logging

import contextily as ctx
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch
from shapely.ops import unary_union

from utils import terminology
from utils.folds import (
    assign_folds,
    assign_spatial_partition,
    connected_components,
    cross_unit_distances,
)
from utils.paths import get_project_paths
from utils.style import get_figure_size, save_figure, use_publication_style

use_publication_style()  # Charis SIL publication typeface (utils.style)

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s", force=True
)

NOTEBOOK = "008_fold_bootstrap_assignment"
PLOT_EPSG = (
    3857  # Web Mercator, for the contextily basemap (display only; areas/distances use 3035)
)

paths = get_project_paths()
labels = gpd.read_file(paths.labels / "ogf_reference_labels.gpkg", layer="parcels")
labels["ogf_group"] = np.select(
    [
        labels["class"].astype(str).str.startswith("OGF"),
        labels["class"].astype(str).str.startswith("Non_OGF"),
    ],
    ["OGF", "Non-OGF"],
    default="Unlabelled",
)
labelled = labels[labels["ogf_group"].isin(["OGF", "Non-OGF"])].copy()
ogf = labelled["ogf_group"].eq("OGF")

components = connected_components(labelled)
comp_sizes = components.value_counts()
print(f"[load] {len(labels):,} parcels; {len(labelled):,} labelled ({int(ogf.sum()):,} old-growth)")
print(
    f"[components] {comp_sizes.size:,} connected components among labelled parcels; "
    f"the largest {comp_sizes.max():,} is comprised of parcels"
)

## Partition configuration

In [ ]:
# Fold configuration (labelled parcels, six folds, 3x2 grid).
FOLD_OBJECTIVE = "share"  # "share" balances old-growth area; "ratio" balances prevalence
FOLD_PRIORITY = "joint"  # "joint" or "primary_first"
FOLD_PREVALENCE_BASIS = "count"  # only used when FOLD_OBJECTIVE == "ratio"
FOLD_BUFFER_M = 0.0  # raise to merge nearby components and widen fold separation
FOLD_TOLERANCE = 0.25
FOLD_ROTATIONS = tuple(range(0, 180, 5))
# Prevalence experiment (raises at 0.25 on this data; relax tolerance to inspect):
# FOLD_OBJECTIVE, FOLD_PRIORITY, FOLD_TOLERANCE = "ratio", "primary_first", 0.6

# Bootstrap-unit layout (cols, rows); the count is pinned in terminology. The grid is
# fixed to bound the area search; coarse rotations bound runtime. group_connected=False,
# area only, no prevalence balance. One labelled-parcel block bootstrap serves both the
# performance bootstrap and the predicted-area uncertainty.
BOOTSTRAP_GRID = (5, 4)
assert BOOTSTRAP_GRID[0] * BOOTSTRAP_GRID[1] == terminology.BOOTSTRAP_N_UNITS
BOOTSTRAP_ROTATIONS = tuple(range(0, 180, 30))
BOOTSTRAP_TOLERANCE = 1.0  # effectively "report, do not reject"

OUT_GPKG = paths.labels / "ogf_reference_labels_partitioned.gpkg"

## Assign folds and bootstrap units

Adds two columns to the labelled parcels: `fold_id` and `bootstrap_id`.

In [ ]:
# Assign the two partition columns and write them back onto the full labels frame.
fold_kwargs = dict(
    secondary_objective=FOLD_OBJECTIVE,
    priority=FOLD_PRIORITY,
    prevalence_basis=FOLD_PREVALENCE_BASIS,
    buffer_m=FOLD_BUFFER_M,
    tolerance=FOLD_TOLERANCE,
    rotations=FOLD_ROTATIONS,
)
fold_id = assign_folds(labelled, ogf, **fold_kwargs)

bootstrap_id = assign_spatial_partition(
    labelled,
    terminology.BOOTSTRAP_N_UNITS,
    grid_shape=BOOTSTRAP_GRID,
    group_connected=False,
    rotations=BOOTSTRAP_ROTATIONS,
    tolerance=BOOTSTRAP_TOLERANCE,
)

labels["fold_id"] = pd.Series(fold_id, index=labelled.index).reindex(labels.index).astype("Int64")
labels["bootstrap_id"] = (
    pd.Series(bootstrap_id, index=labelled.index).reindex(labels.index).astype("Int64")
)

labels.to_file(OUT_GPKG, driver="GPKG")
print(f"[write] {OUT_GPKG.name}: {len(labels):,} parcels, two partition columns")
print(f"[folds] {fold_id.nunique()} folds; [bootstrap] {bootstrap_id.nunique()} units")

## Statistical analysis: per-unit balance

Total area, old-growth area and prevalence per unit, with the min-max ratios.

In [ ]:
# Per-unit balance table: parcels, total and old-growth area (ha), prevalence (% of parcels).
def balance_table(frame, unit_col, *, include_ogf=True):
    sub = frame[frame[unit_col].notna()].copy()
    sub[unit_col] = sub[unit_col].astype(int)
    sub["area_ha"] = sub.geometry.area / 1e4
    is_ogf = sub["ogf_group"].eq("OGF")
    grp = sub.groupby(unit_col)
    tbl = pd.DataFrame({"parcels": grp.size(), "area_ha": grp["area_ha"].sum()})
    if include_ogf:
        tbl["ogf_ha"] = (
            sub.loc[is_ogf].groupby(unit_col)["area_ha"].sum().reindex(tbl.index).fillna(0.0)
        )
        tbl["n_ogf"] = (
            sub.loc[is_ogf].groupby(unit_col).size().reindex(tbl.index).fillna(0).astype(int)
        )
        tbl["prevalence_pct"] = 100.0 * tbl["n_ogf"] / tbl["parcels"]
    return tbl.round(1)


for name, col, inc in [
    ("fold", "fold_id", True),
    ("bootstrap", "bootstrap_id", True),
]:
    tbl = balance_table(labels, col, include_ogf=inc)
    area_ratio = tbl["area_ha"].max() / tbl["area_ha"].min()
    print(f"\n[{name}] {len(tbl)} units; total-area min-max {area_ratio:.2f}")
    if inc:
        ogf_ratio = tbl["ogf_ha"].max() / max(tbl["ogf_ha"].min(), 1e-9)
        print(
            f"[{name}] old-growth min-max {ogf_ratio:.2f}; "
            f"prevalence {tbl['prevalence_pct'].min():.1f}-{tbl['prevalence_pct'].max():.1f}%"
        )
    print(tbl.to_string())

In [ ]:
# Balance bars: total area (and old-growth area, where defined) per unit, with target and band.
def plot_balance(frame, unit_col, title, *, include_ogf=True, tolerance=0.25, skip_target_cols=()):
    tbl = balance_table(frame, unit_col, include_ogf=include_ogf)
    units = tbl.index.to_numpy()
    panels = [("area_ha", "Total area (ha)", "steelblue")]
    if include_ogf:
        panels.append(("ogf_ha", "Old-growth area (ha)", terminology.SEMANTIC_COLOURS["ogf"]))
    fig, axes = plt.subplots(1, len(panels), figsize=get_figure_size("double", aspect=0.4))
    axes = np.atleast_1d(axes)
    for ax, (col, ylab, colour) in zip(axes, panels, strict=True):
        target = tbl[col].sum() / len(tbl)
        ax.bar(units, tbl[col], color=colour, edgecolor="black", linewidth=0.4)
        if col not in skip_target_cols:
            ax.axhline(target, color="red", linestyle="--", linewidth=0.8, label="Target")
            ax.axhspan(
                target * (1 - tolerance),
                target * (1 + tolerance),
                color="red",
                alpha=0.08,
                label=f"{tolerance:.0%} band",
            )
        ax.set_xlabel({"fold_id": "Fold", "bootstrap_id": "Block"}[unit_col])
        ax.set_ylabel(ylab)
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize=7, frameon=False)
        ax.spines[["top", "right"]].set_visible(False)
    fig.suptitle(title, y=0.99)
    fig.tight_layout()
    # Manuscript figures S3 (folds) and S4 (bootstrap blocks).
    figure = {"fold_id": "fig_s3_balance_fold_id", "bootstrap_id": "fig_s4_balance_bootstrap_id"}
    save_figure(fig, f"{NOTEBOOK}/{figure[unit_col]}", data=tbl.reset_index())
    plt.show()


plot_balance(labels, "fold_id", "Fold balance", include_ogf=True, tolerance=FOLD_TOLERANCE)
plot_balance(
    labels, "bootstrap_id", "Bootstrap balance", include_ogf=True, skip_target_cols=("ogf_ha",)
)

## Statistical analysis: cross-unit separation distances

Polygon-to-polygon distances between each unit and the others. Full pairwise and nearest-neighbour, by population.

In [ ]:
# Cross-unit distance quantiles (km). Re-derive the labelled subsets FROM the assigned
# `labels` frame so they carry the new partition columns (the earlier subsets predate the
# assignment). Pairwise reproduces the manuscript fold p50/p05; nearest-neighbour shows
# residual proximity that motivates the buffer analysis.
labelled = labels[labels["ogf_group"].isin(["OGF", "Non-OGF"])].copy()
ogf_gdf = labelled[labelled["ogf_group"].eq("OGF")].copy()
non_gdf = labelled[labelled["ogf_group"].eq("Non-OGF")].copy()


def distance_tables(frame, unit_col, *, modes=("pairwise", "nearest")):
    for mode in modes:
        km = cross_unit_distances(frame, unit_col, mode=mode).copy()
        for c in ("min", "q05", "q25", "q50", "q75", "q95"):
            km[c] = (km[c] / 1000.0).round(2)
        print(f"  [{mode}] (km)")
        print(km[["n", "min", "q05", "q25", "q50", "q75", "q95"]].to_string())


for label, frame in [
    ("fold: all labelled", labelled),
    ("fold: old-growth", ogf_gdf),
    ("fold: non-old-growth", non_gdf),
]:
    print(f"\n{label}")
    distance_tables(frame, "fold_id")

for label, frame in [
    ("bootstrap: all labelled", labelled),
    ("bootstrap: old-growth", ogf_gdf),
    ("bootstrap: non-old-growth", non_gdf),
]:
    print(f"\n{label}")
    distance_tables(frame, "bootstrap_id")

In [ ]:
# Within-unit spatial extent (block size). Minimum convex-hull width via rotating
# callipers is the dimension that must exceed the autocorrelation range for a block
# bootstrap. Caveat: for the labelled (fold, bootstrap) partitions the hull spans
# inter-cluster gaps, so min width here is an upper bound on block compactness.
AUTOCORR_RANGE_KM = 6.7


def block_extent(frame, unit_col):
    sub = frame[frame[unit_col].notna()].copy()
    sub[unit_col] = sub[unit_col].astype(int)
    rows = []
    for unit, geoms in sub.groupby(unit_col).geometry:
        coords = np.asarray(unary_union(geoms.values).convex_hull.exterior.coords[:-1])[:, :2]
        m = len(coords)
        min_w, max_d = np.inf, 0.0
        for i in range(m):
            edge = coords[(i + 1) % m] - coords[i]
            length = float(np.hypot(*edge))
            if length == 0:
                continue
            normal = np.array([-edge[1], edge[0]]) / length
            proj = coords @ normal
            width = float(proj.max() - proj.min())
            along = coords @ (edge / length)
            min_w = min(min_w, width)
            max_d = max(max_d, max(width, float(along.max() - along.min())))
        rows.append((unit, min_w / 1000.0, max_d / 1000.0))
    return (
        pd.DataFrame(rows, columns=[unit_col, "min_width_km", "max_diameter_km"])
        .set_index(unit_col)
        .round(1)
    )


for name, col in [
    ("fold", "fold_id"),
    ("bootstrap", "bootstrap_id"),
]:
    ext = block_extent(labels, col)
    below = int((ext["min_width_km"] < AUTOCORR_RANGE_KM).sum())
    print(
        f"\n[{name}] block extent (km); {below}/{len(ext)} units below the "
        f"{AUTOCORR_RANGE_KM} km range"
    )
    print(
        f"  min-width range {ext['min_width_km'].min():.1f}-{ext['min_width_km'].max():.1f}; "
        f"median {ext['min_width_km'].median():.1f}"
    )
    print(ext.to_string())

In [ ]:
# Cross-unit separation boxplots. One 1x3 panel (all / old-growth / non-old-growth) per
# labelled partition using the full pairwise distances; whiskers span the full range.
def separation_panel(subsets, unit_col, mode, suptitle, fname):
    fig, axes = plt.subplots(1, len(subsets), figsize=get_figure_size("double", aspect=0.32))
    axes = np.atleast_1d(axes)
    records = []
    for ax, (label, frame) in zip(axes, subsets.items(), strict=True):
        _, arrays = cross_unit_distances(frame, unit_col, mode=mode, return_arrays=True)
        units = sorted(arrays)
        ax.boxplot(
            [arrays[u] / 1000.0 for u in units],
            tick_labels=[str(u) for u in units],
            whis=(0, 100),
            showfliers=False,
        )
        for u in units:
            records.extend(
                {"subset": label, "unit": int(u), "distance_km": float(d) / 1000.0}
                for d in arrays[u]
            )
        ax.set_xlabel("Unit")
        ax.set_ylabel("Distance (km)")
        ax.set_title(label)
        ax.grid(True, axis="y", alpha=0.3)
        ax.spines[["top", "right"]].set_visible(False)
    fig.suptitle(suptitle, y=0.99)
    fig.tight_layout()
    save_figure(fig, f"{NOTEBOOK}/{fname}", data=pd.DataFrame.from_records(records))
    plt.show()


labelled_subsets = {"All labelled": labelled, "Old-growth": ogf_gdf, "Non-old-growth": non_gdf}
separation_panel(
    labelled_subsets, "fold_id", "pairwise", "Cross-fold separation (km)", "separation_fold"
)
separation_panel(
    labelled_subsets,
    "bootstrap_id",
    "pairwise",
    "Cross-unit separation: bootstrap (km)",
    "separation_bootstrap",
)

## Partition maps

Parcels coloured by unit over an OpenStreetMap basemap; parcels with no unit (unlabelled) shown in grey.

In [ ]:
# Map parcels coloured by unit over a basemap (display in Web Mercator; areas stay on 3035).
def unit_palette(unit_ids):
    ids = sorted(int(u) for u in unit_ids)
    order = terminology.PALETTE_CATEGORICAL_ORDER
    if len(ids) <= len(order):
        return {u: order[i] for i, u in enumerate(ids)}
    cmap = plt.get_cmap("hsv")
    return {u: cmap(i / len(ids)) for i, u in enumerate(ids)}


def plot_partition_map(frame, unit_col, title):
    disp = frame.to_crs(epsg=PLOT_EPSG)
    assigned = disp[disp[unit_col].notna()].copy()
    assigned[unit_col] = assigned[unit_col].astype(int)
    palette = unit_palette(assigned[unit_col].unique())
    fig, ax = plt.subplots(figsize=(12, 9))
    unassigned = disp[disp[unit_col].isna()]
    if len(unassigned):
        unassigned.plot(ax=ax, facecolor="lightgrey", edgecolor="grey", linewidth=0.2, alpha=0.6)
    for u, colour in palette.items():
        assigned[assigned[unit_col] == u].plot(
            ax=ax, color=colour, edgecolor="black", linewidth=0.3, alpha=0.9
        )
    try:
        ctx.add_basemap(ax, source=ctx.providers.Esri.WorldGrayCanvas, attribution_size=6)
    except Exception as exc:
        logging.warning("basemap skipped: %s", exc)

    if len(palette) <= len(terminology.PALETTE_CATEGORICAL_ORDER):
        ax.legend(
            handles=[
                Patch(facecolor=c, edgecolor="black", label=f"{unit_col.split('_')[0].title()} {u}")
                for u, c in palette.items()
            ],
            loc="lower left",
            frameon=True,
            framealpha=0.9,
            fontsize=8,
        )
    ax.set_axis_off()
    ax.set_title(title)
    fig.tight_layout()
    save_figure(fig, f"{NOTEBOOK}/map_{unit_col}", data=assigned[["parcel_id", unit_col]])
    plt.show()


plot_partition_map(labels, "fold_id", "Spatial folds (1-6)")
plot_partition_map(labels, "bootstrap_id", "Bootstrap units")

## Fold buffer sensitivity

For each fold held out in turn, the training area (all, old-growth, non-old-growth) retained as a no-train buffer around the held-out fold widens.

In [ ]:
# Fold buffer sensitivity: treat each fold as the held-out set and measure how much
# training area survives a buffer of increasing radius around it (Ploton-style exclusion).
BUFFER_M = np.arange(0, 20_001, 5000, dtype=float)
fold_ids = sorted(int(f) for f in labels["fold_id"].dropna().unique())
folded = labels[labels["fold_id"].notna()].copy()
folded["fold_id"] = folded["fold_id"].astype(int)
folded["area_m2"] = folded.geometry.area
folded["is_ogf"] = folded["ogf_group"].eq("OGF")


def buffer_curve(fold_id):
    val = folded[folded["fold_id"] == fold_id]
    train = folded[folded["fold_id"] != fold_id]
    base = {
        k: float(train.loc[m, "area_m2"].sum())
        for k, m in {
            "all": train.index,
            "ogf": train.index[train["is_ogf"]],
            "non": train.index[~train["is_ogf"]],
        }.items()
    }
    val_union = unary_union(val.geometry.values)
    sidx = train.sindex
    rem = {k: np.full(len(BUFFER_M), v) for k, v in base.items()}
    for i, d in enumerate(BUFFER_M[1:], start=1):
        buf = val_union.buffer(float(d))
        hit = train.iloc[list(sidx.intersection(buf.bounds))]
        hit = hit[hit.geometry.intersects(buf)]
        rem["all"][i] = base["all"] - float(hit["area_m2"].sum())
        rem["ogf"][i] = base["ogf"] - float(hit.loc[hit["is_ogf"], "area_m2"].sum())
        rem["non"][i] = base["non"] - float(hit.loc[~hit["is_ogf"], "area_m2"].sum())
    return base, rem


curves = {f: buffer_curve(f) for f in fold_ids}

# Buffer table: training area retained (%) per fold and the average, at each buffer.
buffer_rows = []
for i, d in enumerate(BUFFER_M):
    row = {"buffer_km": d / 1000.0}
    for key, label in [("all", "all"), ("ogf", "ogf"), ("non", "non")]:
        pct = np.array([100.0 * curves[f][1][key][i] / curves[f][0][key] for f in fold_ids])
        for f, p in zip(fold_ids, pct, strict=True):
            row[f"{label}_fold{f}"] = round(float(p), 1)
        row[f"{label}_avg"] = round(float(pct.mean()), 1)
    buffer_rows.append(row)
buffer_tbl = pd.DataFrame(buffer_rows).set_index("buffer_km")
print("\n[buffer] training area retained (%), average across folds:")
print(buffer_tbl[["all_avg", "ogf_avg", "non_avg"]].to_string())
print("\n[buffer] all-parcels retained (%) per fold:")
print(buffer_tbl[[f"all_fold{f}" for f in fold_ids]].to_string())

fig, axes = plt.subplots(1, 3, figsize=get_figure_size("double", aspect=0.34))
palette = unit_palette(fold_ids)
for ax, (key, label) in zip(
    axes, [("all", "All"), ("ogf", "Old-growth"), ("non", "Non-old-growth")], strict=True
):
    avg = np.zeros(len(BUFFER_M))
    for f in fold_ids:
        base, rem = curves[f]
        pct = 100.0 * rem[key] / base[key]
        avg += pct
        ax.plot(
            BUFFER_M / 1000.0, pct, color=palette[f], linewidth=1.0, alpha=0.7, label=f"Fold {f}"
        )
    ax.plot(
        BUFFER_M / 1000.0,
        avg / len(fold_ids),
        color="black",
        linewidth=2.0,
        label="Average",
        zorder=10,
    )
    ax.set_xlabel("Buffer distance (km)")
    ax.set_ylabel("Training area retained (%)")
    ax.set_title(label)
    ax.set_ylim(0, 105)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=6, frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
fig.suptitle("Fold buffer sensitivity", y=0.99)
fig.tight_layout()
save_figure(fig, f"{NOTEBOOK}/buffer_sensitivity_fold", data=buffer_tbl.reset_index())
plt.show()

In [ ]:
# Buffer table: training area retained (%) per fold and the average, at each buffer.
buffer_rows = []
for i, d in enumerate(BUFFER_M):
    row = {"buffer_km": d / 1000.0}
    for key, label in [("all", "all"), ("ogf", "ogf"), ("non", "non")]:
        pct = np.array([100.0 * curves[f][1][key][i] / curves[f][0][key] for f in fold_ids])
        for f, p in zip(fold_ids, pct, strict=True):
            row[f"{label}_fold{f}"] = round(float(p), 1)
        row[f"{label}_avg"] = round(float(pct.mean()), 1)
    buffer_rows.append(row)
buffer_tbl = pd.DataFrame(buffer_rows).set_index("buffer_km")
print("\n[buffer] training area retained (%), average across folds:")
print(buffer_tbl[["all_avg", "ogf_avg", "non_avg"]].to_string())
print("\n[buffer] all-parcels retained (%) per fold:")
print(buffer_tbl[[f"all_fold{f}" for f in fold_ids]].to_string())